# train_gcn_snapshot_v3.ipynb
**GCN Snapshot** — VRAM-safe for GTX 1650 4GB

**Runs from:** `backend/model training/`

In [ ]:
import os, pickle, time, warnings, gc, glob
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import scipy.sparse as sp, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore'); sns.set_theme(style='darkgrid')

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME='gcn_snapshot'; MODEL_DISPLAY='GCN Snapshot'
EMBED_DIM=32; ZONE_EMBED_DIM=16; HIDDEN=192
LR=2e-3; WARMUP_EP=5; PATIENCE=15; MAX_EPOCHS=80; WD=1e-4

DATA_DIR='../data generation/data'
PATTERN_DIR=f'{DATA_DIR}/pattern_features'; SCALER_DIR=f'{DATA_DIR}/scalers'
PROC_DIR=f'{DATA_DIR}/processed'; ADJ_DIR=f'{DATA_DIR}/graph_adj'
MODEL_DIR=f'../models/{MODEL_NAME}'; os.makedirs(MODEL_DIR, exist_ok=True)
def _save(o,p): torch.save(o,str(Path(p).resolve()))
def _load(p,**k): return torch.load(str(Path(p).resolve()),**k)

with open(f'{PATTERN_DIR}/pattern_meta.pkl','rb') as f: meta=pickle.load(f)
with open(f'{SCALER_DIR}/scaler_pattern_ttr.pkl','rb') as f: scaler_ttr=pickle.load(f)
with open(f'{SCALER_DIR}/scaler_pattern_cong.pkl','rb') as f: scaler_cong=pickle.load(f)
with open(f'{ADJ_DIR}/corridor_meta.pkl','rb') as f: corridor_meta=pickle.load(f)
FEATURES=meta['pattern_features']; NF=meta['n_features']; NE=meta['n_edges']
df_st=pd.read_parquet(f'{PROC_DIR}/edges_static_scaled.parquet')
N_ZONES=int(df_st['zone_id'].max())+1; corridors=sorted(corridor_meta.keys())

edge_info={}
for cid in corridors:
    cm=corridor_meta[cid]
    for eid,li in cm['local_map'].items():
        gi=cm['global_indices'][li]
        m=df_st['edge_id']==eid; zi=int(df_st.loc[m,'zone_id'].iloc[0]) if m.sum()>0 else 0
        edge_info[eid]={'cid':cid,'li':li,'gi':gi,'zi':zi}
del df_st; gc.collect()

batch_files=sorted(glob.glob(f'{PATTERN_DIR}/batch_*.parquet'))
print(f'Features: {NF}, Edges: {NE:,}, Zones: {N_ZONES}, Corridors: {len(corridors)}, Batches: {len(batch_files)}')

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self,ic,oc):
        super().__init__(); self.lin=nn.Linear(ic,oc); self.norm=nn.LayerNorm(oc)
        self.res=nn.Linear(ic,oc) if ic!=oc else nn.Identity()
    def forward(self,x,A):
        h=torch.sparse.mm(A,x) if A.is_sparse else torch.mm(A,x)
        return self.norm(F.relu(self.lin(h))+self.res(x))

class GCNSnapshot(nn.Module):
    def __init__(self,nc,ne,nz,ed=32,zd=16,hid=192):
        super().__init__(); self.ee=nn.Embedding(ne,ed); self.ze=nn.Embedding(nz,zd)
        ind=nc+ed+zd; self.g1=GCNLayer(ind,hid); self.g2=GCNLayer(hid,hid); self.g3=GCNLayer(hid,hid)
        self.fc=nn.Linear(hid,1)
    def forward(self,x,ei,zi,A):
        h=torch.cat([x,self.ee(ei),self.ze(zi)],dim=-1); h=self.g1(h,A); h=self.g2(h,A); h=self.g3(h,A)
        return self.fc(h).squeeze(-1)

def load_adj(cid,N):
    with open(f'{ADJ_DIR}/corridor_{cid:04d}.pkl','rb') as f: ad=pickle.load(f)
    r,c,v=ad['A_fwd_coo']
    r2=np.concatenate([r,np.arange(N,dtype=np.int32)]); c2=np.concatenate([c,np.arange(N,dtype=np.int32)])
    v2=np.concatenate([v,np.ones(N,dtype=np.float32)]); A=sp.coo_matrix((v2,(r2,c2)),shape=(N,N)).tocsr()
    d=np.array(A.sum(1)).flatten(); d[d==0]=1; An=sp.diags(1/d)@A
    Ac=An.tocoo(); i=torch.stack([torch.from_numpy(Ac.row.astype(np.int64)),torch.from_numpy(Ac.col.astype(np.int64))])
    return torch.sparse_coo_tensor(i,torch.from_numpy(Ac.data.astype(np.float32)),(N,N)).coalesce().to(device)

def snap_populate(snap_df, N, feat_cols, tcol, dev):
    eids = snap_df['edge_id'].values
    lis, gis, zis, valid = [], [], [], []
    for i, eid in enumerate(eids):
        info = edge_info.get(eid)
        if info is None or info['li'] >= N: continue
        lis.append(info['li']); gis.append(info['gi']); zis.append(info['zi']); valid.append(i)
    if len(lis) < 3: return None, None, None, None, None
    lis_a = np.array(lis, dtype=np.int64)
    xc=torch.zeros(N,len(feat_cols),device=dev); yt=torch.zeros(N,device=dev)
    ei=torch.zeros(N,dtype=torch.long,device=dev); zi=torch.zeros(N,dtype=torch.long,device=dev)
    mask=torch.zeros(N,dtype=torch.bool,device=dev)
    xc[lis_a]=torch.from_numpy(snap_df.iloc[valid][feat_cols].values.astype(np.float32)).to(dev)
    yt[lis_a]=torch.from_numpy(snap_df.iloc[valid][tcol].values.astype(np.float32)).to(dev)
    ei[lis_a]=torch.tensor(gis,dtype=torch.long,device=dev)
    zi[lis_a]=torch.tensor(zis,dtype=torch.long,device=dev); mask[lis_a]=True
    return xc, yt, ei, zi, mask
print('Model defined.')

In [ ]:
def train_gcn(tcol, sname, tname):
    print(f'\n{"="*50}\n  GCN Snapshot — {tname}\n{"="*50}')
    model=GCNSnapshot(NF,NE,N_ZONES,EMBED_DIM,ZONE_EMBED_DIM,HIDDEN).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,'min',factor=.5,patience=7,min_lr=1e-6)
    crit=nn.SmoothL1Loss(); best_v=float('inf'); pat=0; hist={'train_loss':[],'val_loss':[]}
    print(f'  Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
    for epoch in range(MAX_EPOCHS):
        te=time.perf_counter()
        for pg in opt.param_groups: pg['lr']=LR*min(1,(epoch+1)/WARMUP_EP)
        model.train(); eloss=0; nsn=0
        for bf in np.random.permutation(batch_files):
            df=pd.read_parquet(bf); tr=df[df['split']=='train']; del df
            if len(tr)<10: del tr; continue
            for (cid,hr), grp_idx in tr.groupby(['corridor_id','hour_index']).groups.items():
                cm=corridor_meta.get(int(cid))
                if cm is None or cm['N']<3: continue
                N=cm['N']; snap=tr.loc[grp_idx]
                if len(snap)<3: continue
                A=load_adj(int(cid),N)
                xc,yt,ei,zi,mask=snap_populate(snap,N,FEATURES,tcol,device)
                if xc is None: continue
                opt.zero_grad(); loss=crit(model(xc,ei,zi,A)[mask],yt[mask])
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.); opt.step()
                eloss+=loss.item(); nsn+=1
            del tr; gc.collect(); torch.cuda.empty_cache()
        tl=eloss/max(nsn,1)
        model.eval(); vloss=0; vn=0
        with torch.no_grad():
            for bf in batch_files[::3]:
                df=pd.read_parquet(bf); vd=df[df['split']=='val']; del df
                if len(vd)<10: del vd; continue
                for (cid,hr), gi in vd.groupby(['corridor_id','hour_index']).groups.items():
                    cm=corridor_meta.get(int(cid))
                    if cm is None or cm['N']<3: continue
                    N=cm['N']; snap=vd.loc[gi]
                    if len(snap)<3: continue
                    A=load_adj(int(cid),N)
                    xc,yt,ei,zi,mask=snap_populate(snap,N,FEATURES,tcol,device)
                    if xc is None: continue
                    vloss+=crit(model(xc,ei,zi,A)[mask],yt[mask]).item(); vn+=1
                del vd; gc.collect()
        vl=vloss/max(vn,1); hist['train_loss'].append(tl); hist['val_loss'].append(vl)
        if epoch>=WARMUP_EP: sched.step(vl)
        mk=''
        if vl<best_v: best_v=vl; pat=0; _save(model.state_dict(),f'{MODEL_DIR}/{sname}'); mk=' ★'
        else: pat+=1
        print(f'  Ep {epoch+1:3d} | trn={tl:.5f} val={vl:.5f} | {time.perf_counter()-te:.0f}s{mk}')
        if pat>=PATIENCE: print('  Early stop'); break
        torch.cuda.empty_cache(); gc.collect()
    model.load_state_dict(_load(f'{MODEL_DIR}/{sname}',weights_only=True)); return model, hist

model_ttr, hist_ttr = train_gcn('travel_time_ratio','best_ttr.pt','TTR')
model_ttr.cpu(); torch.cuda.empty_cache(); gc.collect()
model_cong, hist_cong = train_gcn('congestion_level','best_cong.pt','Congestion')
model_ttr.to(device)
print(f'\n✓ Both saved to {MODEL_DIR}/')

In [ ]:
print('Test predictions...')
all_tp,all_ta,all_cp,all_ca,all_ff,all_h=[],[],[],[],[],[]
for bf in batch_files:
    df=pd.read_parquet(bf); te=df[df['split']=='test']
    if len(te)==0: del df; continue
    for (cid,hr), gi in te.groupby(['corridor_id','hour_index']).groups.items():
        cm=corridor_meta.get(int(cid))
        if cm is None or cm['N']<3: continue
        N=cm['N']; snap=te.loc[gi]; A=load_adj(int(cid),N)
        eids=snap['edge_id'].values; lis,gis,zis,valid=[],[],[],[]
        for i,eid in enumerate(eids):
            info=edge_info.get(eid)
            if info is None or info['li']>=N: continue
            lis.append(info['li']); gis.append(info['gi']); zis.append(info['zi']); valid.append(i)
        if len(lis)<3: continue
        lis_a=np.array(lis,dtype=np.int64); valid_snap=snap.iloc[valid]
        xc=torch.zeros(N,NF,device=device); ei=torch.zeros(N,dtype=torch.long,device=device); zi=torch.zeros(N,dtype=torch.long,device=device)
        xc[lis_a]=torch.from_numpy(valid_snap[FEATURES].values.astype(np.float32)).to(device)
        ei[lis_a]=torch.tensor(gis,dtype=torch.long,device=device); zi[lis_a]=torch.tensor(zis,dtype=torch.long,device=device)
        with torch.no_grad():
            pred_t=model_ttr(xc,ei,zi,A).cpu().numpy(); pred_c=model_cong(xc,ei,zi,A).cpu().numpy()
        for i,_ in enumerate(valid):
            row=valid_snap.iloc[i]; li=lis[i]
            all_tp.append(pred_t[li]); all_ta.append(row['travel_time_ratio'])
            all_cp.append(pred_c[li]); all_ca.append(row['congestion_level'])
            all_ff.append(row.get('static_free_flow_travel_time',30.0)); all_h.append(row.get('hour_of_day',0))
    del df,te; gc.collect()
n=len(all_tp)
ttr_pred_inv=np.array(all_tp)*scaler_ttr.scale_[0]+scaler_ttr.mean_[0]; ttr_act_inv=np.array(all_ta)*scaler_ttr.scale_[0]+scaler_ttr.mean_[0]
cong_pred_inv=np.array(all_cp)*scaler_cong.scale_[0]+scaler_cong.mean_[0]; cong_act_inv=np.array(all_ca)*scaler_cong.scale_[0]+scaler_cong.mean_[0]
fftt_test=np.array(all_ff); hour_arr=np.array(all_h)
ttr_pred_sec=ttr_pred_inv*fftt_test; ttr_act_sec=ttr_act_inv*fftt_test
for nm,p,a in [('TTR',ttr_pred_inv,ttr_act_inv),('TTR s',ttr_pred_sec,ttr_act_sec),('Cng',cong_pred_inv,cong_act_inv)]:
    print(f'{nm}: MAE={mean_absolute_error(a,p):.4f} R²={r2_score(a,p):.4f}')

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(18,15)); fig.suptitle(f'{MODEL_DISPLAY}',fontsize=16,fontweight='bold')
ax=axes[0,0]; ax.plot(hist_ttr['train_loss'],label='TTR trn',c='steelblue'); ax.plot(hist_ttr['val_loss'],label='TTR val',c='steelblue',ls='--')
ax.plot(hist_cong['train_loss'],label='Cng trn',c='coral'); ax.plot(hist_cong['val_loss'],label='Cng val',c='coral',ls='--'); ax.legend(fontsize=8); ax.set_yscale('log')
ns=min(10000,len(ttr_pred_sec)); ii=np.random.choice(ns,ns,replace=False)
axes[0,1].scatter(ttr_act_sec[ii],ttr_pred_sec[ii],alpha=.12,s=3,c='steelblue')
axes[0,2].scatter(cong_act_inv[ii],cong_pred_inv[ii],alpha=.12,s=3,c='coral')
te=ttr_pred_inv-ttr_act_inv; ce=cong_pred_inv-cong_act_inv
sns.histplot(te[np.abs(te)<np.percentile(np.abs(te),99)],kde=True,ax=axes[1,0],color='steelblue',bins=50)
axes[1,1].scatter(ttr_pred_inv[ii],te[ii],alpha=.1,s=3,c='steelblue'); axes[1,1].axhline(0,c='r',ls='--')
ta=np.sort(np.abs(te)); ca=np.sort(np.abs(ce)); axes[1,2].plot(ta,np.linspace(0,1,len(ta)),label='TTR')
axes[1,2].plot(ca,np.linspace(0,1,len(ca)),label='Cng'); axes[1,2].legend()
axes[2,0].text(.5,.5,'Per-corridor model',ha='center',va='center',transform=axes[2,0].transAxes)
axes[2,1].text(.5,.5,f'Test: {len(ttr_pred_inv):,}',ha='center',va='center',transform=axes[2,1].transAxes)
axes[2,2].axis('off')
s=[['','TTR','Cng'],['MAE',f'{mean_absolute_error(ttr_act_inv,ttr_pred_inv):.4f}',f'{mean_absolute_error(cong_act_inv,cong_pred_inv):.4f}'],
   ['R²',f'{r2_score(ttr_act_inv,ttr_pred_inv):.4f}',f'{r2_score(cong_act_inv,cong_pred_inv):.4f}']]
axes[2,2].table(cellText=s[1:],colLabels=s[0],loc='center',cellLoc='center')
plt.tight_layout(); plt.savefig(f'{MODEL_DIR}/eval.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'✓ {MODEL_DISPLAY} complete.')